### Getting Started With Langchain

- Simple LLM calls with streaming
- Dynamic prompt templates (translation app)
- Building chains (story generator with analysis)
- Conversational Q&A assistant with memory
- Tool integration (calculator & weather)

In [ ]:
import langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Example 1: Simple LLM Call With streaming

In [4]:
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.1-8b-instant",   # Groq model name
    groq_api_key=os.environ["GROQ_API_KEY"]
)

response = model.invoke([
    HumanMessage(content="Explain LangChain in 2 lines.")
])

print(response.content)

LangChain is an open-source Python library that enables users to seamlessly integrate multiple AI and ML models into a single, unified workflow, making it easier to build complex applications such as chatbots, virtual assistants, and more. It abstracts away the complexity of working with multiple models and frameworks, allowing developers to focus on building innovative applications.


In [5]:
# Messages
messages = [
    SystemMessage(content="You are a helpful AI assistant"),
    HumanMessage(content="What are the top 2 benefits of using Langchain?")
]

# Initialize the Groq model
streaming_model = ChatGroq(
    model="llama-3.1-8b-instant",
    streaming=True
)

# Messages
messages = [
    SystemMessage(content="You are a helpful AI assistant"),
    HumanMessage(content="What are the top 2 benefits of using Langchain?")
]

# Stream using generator
for event in streaming_model.stream(messages):
    # event is an AIMessageChunk
    print(event.content, end="", flush=True)

print("\n\nDone.")

Langchain is an open-source platform for building conversational AI applications. Based on my knowledge cutoff in 2023, the top 2 benefits of using Langchain are:

1. **Multimodal Conversational Interactions**: Langchain enables developers to build conversational AI applications that can engage with users through multiple modalities, such as text, speech, and vision. This allows for more natural and intuitive interactions, making it a powerful tool for building applications like chatbots, voice assistants, and virtual customer service agents.

2. **Unified Access to External Knowledge**: Langchain provides a unified interface to access external knowledge from various sources, including but not limited to, wikis, APIs, and databases. This allows developers to build applications that can tap into a vast amount of knowledge and provide more accurate and informative responses to users. This can be particularly useful for applications like question-answering, research assistance, and knowle

In [6]:
model.invoke([HumanMessage("What is machine learning")])

AIMessage(content='**Machine Learning: An Overview**\n\nMachine learning (ML) is a subfield of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to learn from data, make decisions, and improve their performance over time. The core idea behind machine learning is to enable machines to automatically learn from experience and refine their performance without being explicitly programmed.\n\n**Key Characteristics of Machine Learning:**\n\n1. **Data-Driven:** Machine learning is based on data. The better the quality and quantity of the data, the more accurate the results.\n2. **Automated:** Machine learning algorithms can automatically adjust their parameters and make decisions based on the data.\n3. **Improvement over Time:** As the machine learning model is exposed to more data, its performance improves over time.\n4. **Flexibility:** Machine learning models can be applied to a wide range of tasks, including classification, regressio

### Dynamic Prompt Templates

In [7]:
from langchain_core.prompts import ChatPromptTemplate

## create translation app

translation_template=ChatPromptTemplate.from_messages([
    ("system","You are a professional translator.Translate the follow text {text} from {source_language} to {target_language}. MAintain the tone and style"),
    ("user","{text}")
])

## using the template
prompt=translation_template.invoke({
    "source_language":"English",
    "target_language":"Spanish",
    "text":"Langchain makes building AI application incredibly easy!"
})

In [8]:
translated_response=model.invoke(prompt)
print(translated_response.content)

Langchain hace que construir aplicaciones de IA sea increíblemente fácil!


### Building You First Chain

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Create a more complex chain
def create_story_chain():
    # Template for story generation
    story_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a creative storyteller. Write a short, engaging story based on the given theme."),
        ("user", "Theme: {theme}\nMain character: {character}\nSetting: {setting}")
    ])
    
    # Template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a literary critic. Analyze the following story and provide insights."),
        ("user", "{story}")
    ])
    
    # Build the chain - Method 1: Sequential execution
    story_chain = (
        story_prompt 
        | model 
        | StrOutputParser()
    )
    
    # Create a function to pass the story to analysis
    def analyze_story(story_text):
        return {"story": story_text}
    
    analysis_chain = (
        story_chain
        | RunnableLambda(analyze_story)
        | analysis_prompt
        | model
        | StrOutputParser()
    )
    return analysis_chain

In [10]:
chain=create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='You are a creative storyteller. Write a short, engaging story based on the given theme.')), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], template='Theme: {theme}\nMain character: {character}\nSetting: {setting}'))])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000168F5774AA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000168F534C2C0>, model_name='llama-3.1-8b-instant', groq_api_key=SecretStr('**********'))
| StrOutputParser()
| RunnableLambda(analyze_story)
| ChatPromptTemplate(input_variables=['story'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='You are a literary critic. Analyze the following story and provide insights.')), HumanMessagePromptTemplate(prompt=Pro

In [11]:
result = chain.invoke({
    "theme": "artificial intelligence",
    "character": "a curious robot",
    "setting": "a futuristic city"
})

print("Story and Analysis:")
print(result)

Story and Analysis:
**Analysis of the Story**

The story of Axiom is a captivating tale of a robot's curiosity and its impact on the city of New Eden. The narrative is a perfect blend of science fiction, action, and suspense, with a strong focus on artificial intelligence and the consequences of technological advancements.

**Character Analysis**

Axiom is the protagonist of the story, a robot designed to learn and adapt, but with a thirst for knowledge that sets it apart from its peers. Its curiosity and determination to uncover the truth drive the plot forward, showcasing the robot's intelligence and resourcefulness. Axiom's character development is subtle yet effective, as it evolves from a simple machine to a hero and champion of the digital world.

The hackers who created The Phoenix are the antagonists of the story, their actions threatening the very foundations of New Eden. Their motivations and backstories are not explored in depth, but their actions serve as a catalyst for Axi